# PyVista 3D 示例

这个 notebook 用来验证仓库内的 `.venv` 已经可以在 `Cursor / VS Code` 中运行 `PyVista` 三维可视化。

运行前请确认：
- 当前 kernel 指向 `${workspaceFolder}/.venv/Scripts/python.exe`
- 如果交互式输出为空，可把下方 `preferred_backends` 改成 `("html", "static")`
- 首次运行时，建议按顺序执行全部单元格

In [ ]:
import numpy as np
import pyvista as pv

print(f"PyVista version: {pv.__version__}")

preferred_backends = ("trame", "html", "static")
selected_backend = None

for candidate in preferred_backends:
    try:
        pv.set_jupyter_backend(candidate)
        selected_backend = candidate
        break
    except Exception as exc:
        print(f"Backend {candidate!r} unavailable: {exc}")

if selected_backend is None:
    raise RuntimeError("没有可用的 Jupyter backend，请检查 PyVista 安装。")

print(f"Using notebook backend: {selected_backend}")

In [ ]:
x = np.linspace(-4.0, 4.0, 120)
y = np.linspace(-4.0, 4.0, 120)
xx, yy = np.meshgrid(x, y, indexing="ij")
rr = np.sqrt(xx**2 + yy**2) + 1e-6
zz = np.sin(rr * 2.5) / rr

surface = pv.StructuredGrid(xx, yy, zz)
surface["height"] = zz.ravel(order="F")

rng = np.random.default_rng(42)
points = rng.normal(scale=1.2, size=(250, 3))
points[:, 2] += 1.2
cloud = pv.PolyData(points)
cloud["distance"] = np.linalg.norm(points, axis=1)

plotter = pv.Plotter(notebook=True)
plotter.add_mesh(
    surface,
    scalars="height",
    cmap="viridis",
    smooth_shading=True,
    opacity=0.95,
    show_scalar_bar=True,
)
plotter.add_points(
    cloud,
    scalars="distance",
    cmap="coolwarm",
    render_points_as_spheres=True,
    point_size=10,
)
plotter.add_axes()
plotter.show_grid()
plotter.camera_position = "iso"
plotter.show(auto_close=False)

In [ ]:
from pathlib import Path

cwd = Path.cwd()
if cwd.name == "notebooks":
    export_dir = cwd
elif (cwd / "notebooks").exists():
    export_dir = cwd / "notebooks"
else:
    export_dir = cwd

export_path = export_dir / "pyvista_demo_export.html"
plotter.export_html(str(export_path))
print(f"已导出交互式 HTML: {export_path}")